In [20]:
from langgraph.graph import StateGraph, START, END
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from dotenv import load_dotenv
from typing import TypedDict
from langgraph.checkpoint.memory import InMemorySaver

In [21]:
load_dotenv()

llm = HuggingFaceEndpoint(
    repo_id='Qwen/Qwen2.5-7B-Instruct',
    task='text-generation'
)

model = ChatHuggingFace(llm=llm)

In [22]:
class JokeSchema(TypedDict):
    topic : str
    joke : str
    explanation : str

In [23]:
def generate_joke(state : JokeSchema):
    prompt = f'Generate a joke for the {state['topic']}'
    
    response = model.invoke(prompt).content
    
    return {'joke' : response}

In [24]:
def explain_joke(state : JokeSchema):
    prompt = f'Explain the joke {state['joke']}'
    
    response = model.invoke(prompt).content
    
    return {'explanation' : response}

In [25]:
graph = StateGraph(JokeSchema)

graph.add_node("generate_joke", generate_joke)
graph.add_node("explain_joke", explain_joke)

graph.add_edge(START, "generate_joke")
graph.add_edge("generate_joke", "explain_joke")
graph.add_edge("explain_joke", END)

checkpoint = InMemorySaver()

workflow = graph.compile(checkpointer=checkpoint)


In [26]:
initial_state = {
    'topic' : 'Pizza'
}

config = {"configurable" : {"thread_id" : 1}}

final_workflow = workflow.invoke(initial_state, config=config)

In [27]:
final_workflow

{'topic': 'Pizza',
 'joke': "Sure! Here's a pizza joke for you:\n\nWhy did the pizza go to the doctor?\n\nBecause it was feeling a little crust-achy and needed a slice of relief!",
 'explanation': 'Sure! The joke is a play on words and a pun. Here\'s the breakdown:\n\n- **"Why did the pizza go to the doctor?"** This sets up the context of a typical why-did-something-go-to-something-else question, often used in jokes.\n- **"Because it was feeling a little crust-achy and needed a slice of relief!"** This part is where the humor lies:\n  - **"crust-achy"** is a wordplay on "crusty" and "aching."\n  - **"slice of relief"** is a play on "slice of pizza" and the common phrase "slice of relief," which usually means a small amount of comfort or ease in a difficult situation.\n\nThe humor comes from the double entendre and the creative use of words to make a light-hearted joke about a pizza.'}

In [28]:
workflow.get_state(config)

StateSnapshot(values={'topic': 'Pizza', 'joke': "Sure! Here's a pizza joke for you:\n\nWhy did the pizza go to the doctor?\n\nBecause it was feeling a little crust-achy and needed a slice of relief!", 'explanation': 'Sure! The joke is a play on words and a pun. Here\'s the breakdown:\n\n- **"Why did the pizza go to the doctor?"** This sets up the context of a typical why-did-something-go-to-something-else question, often used in jokes.\n- **"Because it was feeling a little crust-achy and needed a slice of relief!"** This part is where the humor lies:\n  - **"crust-achy"** is a wordplay on "crusty" and "aching."\n  - **"slice of relief"** is a play on "slice of pizza" and the common phrase "slice of relief," which usually means a small amount of comfort or ease in a difficult situation.\n\nThe humor comes from the double entendre and the creative use of words to make a light-hearted joke about a pizza.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoi

In [32]:
list(workflow.get_state_history(config))

[StateSnapshot(values={'topic': 'Pizza', 'joke': "Sure! Here's a pizza joke for you:\n\nWhy did the pizza go to the doctor?\n\nBecause it was feeling a little crust-achy and needed a slice of relief!", 'explanation': 'Sure! The joke is a play on words and a pun. Here\'s the breakdown:\n\n- **"Why did the pizza go to the doctor?"** This sets up the context of a typical why-did-something-go-to-something-else question, often used in jokes.\n- **"Because it was feeling a little crust-achy and needed a slice of relief!"** This part is where the humor lies:\n  - **"crust-achy"** is a wordplay on "crusty" and "aching."\n  - **"slice of relief"** is a play on "slice of pizza" and the common phrase "slice of relief," which usually means a small amount of comfort or ease in a difficult situation.\n\nThe humor comes from the double entendre and the creative use of words to make a light-hearted joke about a pizza.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpo